In [ ]:
import csv
import sys
import time

from cryptonets_python_sdk.factor import FaceFactor
from cryptonets_python_sdk.settings.loggingLevel import LoggingLevel
from imutils import paths
from tqdm import tqdm

In [ ]:
server_url = "https://api.cryptonets.ai/node"
api_key = "accsb18b5f17d924db88"

In [ ]:
def list_images(base_path):
    image_types = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
    for (rootDir, dirNames, filenames) in os.walk(base_path):
        for filename in filenames:
            ext = filename[filename.rfind("."):].lower()
            if ext.endswith(image_types):
                imagePath = os.path.join(rootDir, filename)
                yield imagePath

In [ ]:
face_factor = FaceFactor(server_url=server_url, api_key = api_key,logging_level=LoggingLevel.off)

In [ ]:
image_folder_path = "../git_codes/Public/cryptonets-python-sdk/tests/example/test_images/"
image_path_list = list(paths.list_images(image_folder_path))
print("Processing {} images".format(len(image_path_list)))

In [ ]:
with open(f'result_{time.time_ns()}.csv', 'w', newline='') as csvfile:
    fieldnames = ['image_path', 'error', 'message', 'return_code', 'return_message', 'age']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)  
    writer.writeheader() 
    for image_path in tqdm(image_path_list):
        age_handle = face_factor.estimate_age(image_path=image_path)
        age_result = {"image_path":image_path.replace(image_folder_path, ""), "error": age_handle.error, "message": age_handle.message}
        for index, face in enumerate(age_handle.face_objects):
            age_face_result = {"return_code": face.return_code, "return_message": face.message, "age": face.age}
            age_face_result = age_result | age_face_result
            writer.writerow(age_face_result)
            csvfile.flush()
